# 06 · Loss Functions, Gradient Descent and Empirical Risk Minimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anuxlab/deep-learning-systems/blob/main/notebooks/06-losses-gradient-descent/06_losses_gradient_descent.ipynb)
[![Run on GitHub Actions](https://img.shields.io/badge/run-GitHub%20Actions-2088FF?logo=githubactions&logoColor=white)](https://github.com/anuxlab/deep-learning-systems/actions/workflows/nb-06-losses-gradient-descent.yml)

| | |
|---|---|
| **Wiki chapter** | [Module 06](https://github.com/anuxlab/deep-learning-systems/wiki/Module-06) |
| **Level** | Intermediate |
| **Typical CPU runtime** | ~1 min (GitHub free runner, no GPU) |
| **Prerequisites** | Modules 02, 03 |

## What you will do

- Compare 0–1 loss with differentiable surrogates (hinge, logistic, exponential).
- See how MSE, MAE and Huber respond to outliers.
- Derive and verify the gradients of cross-entropy with softmax/sigmoid ($\hat p-y$).
- Run batch, stochastic and mini-batch gradient descent and observe the learning-rate stability threshold $2/L$.
- Measure how feature scaling changes the condition number and convergence speed.

> **How to use this notebook.** Read each markdown cell, *predict* what the next code cell will print, then run it.
> Every section ends with `check(...)` assertions: if one fails, something in your understanding (or the code) is off.
> Figures are saved to `figures/` and a machine-readable summary to `metrics.json` next to this notebook.

## 0 · Setup and reproducibility

We import shared helpers (`show` saves each figure to `figures/`, `check` asserts and logs, `timer` logs wall-clock time), fix all random seeds and print an environment report so every run is traceable (see the wiki's *Implementation and Research Standards*).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "_common"))   # shared helpers: notebooks/_common/dlsys.py
from dlsys import *                                       # np, plt, torch, seed_everything, show, check, timer, ...

SEED = seed_everything(42)
ENV = env_report()

## 1 · 0–1 loss and its convex surrogates
The loss we *care about* is $\mathbb 1[y\,f(x)\le0]$ but it has zero gradient almost everywhere. We optimise a surrogate $\ell(m)$ of the **margin** $m=yf(x)$: hinge $\max(0,1-m)$, logistic $\log(1+e^{-m})$, exponential $e^{-m}$. Each upper-bounds the 0–1 loss (logistic after dividing by $\ln2$).

In [ ]:
m = np.linspace(-3, 3, 601)
zo = (m <= 0).astype(float); hinge = np.maximum(0, 1 - m); logi = np.log1p(np.exp(-m)) / np.log(2); expo = np.exp(-m)
for nme, l in [("hinge", hinge), ("logistic/ln2", logi), ("exponential", expo)]:
    check(np.all(l >= zo - 1e-12), f"{nme} upper-bounds the 0–1 loss")
    plt.plot(m, l, label=nme)
plt.plot(m, zo, "k", label="0–1"); plt.ylim(0, 4); plt.xlabel("margin y·f(x)"); plt.legend(); plt.title("Surrogate losses"); show("surrogate_losses")

## 2 · Regression losses and outliers
MSE is the MLE under Gaussian noise (penalises large residuals quadratically); MAE under Laplace noise (median-like); Huber is quadratic for $|r|\le\delta$ and linear beyond. We fit a line to data with 10% gross outliers using gradient descent on each loss.

In [ ]:
rng = np.random.default_rng(0); n = 100
x = rng.uniform(-1, 1, n); y = 2 * x + 1 + .1 * rng.normal(size=n); out = rng.choice(n, 10, replace=False); y[out] += rng.choice([-8, 8], 10)
Xd = np.c_[x, np.ones(n)]
def fit(loss, lr=.05, steps=3000, delta=.5):
    w = np.zeros(2)
    for _ in range(steps):
        r = Xd @ w - y
        if loss == "mse": g = r
        elif loss == "mae": g = np.sign(r)
        else: g = np.clip(r, -delta, delta)
        w -= lr * Xd.T @ g / n
    return w
res = {l: fit(l) for l in ["mse", "mae", "huber"]}
for l, w in res.items(): print(f"{l:6s} slope={w[0]:.3f} intercept={w[1]:.3f}   (truth 2, 1)")
err = {l: np.linalg.norm(w - [2, 1]) for l, w in res.items()}
check(err["huber"] < err["mse"] and err["mae"] < err["mse"], "robust losses are less affected by outliers than MSE")
plt.scatter(x, y, s=12); tt = np.array([-1, 1])
for l, w in res.items(): plt.plot(tt, w[0] * tt + w[1], label=l)
plt.legend(); plt.title("Line fits with 10% outliers"); show("robust_regression")
METRICS["outlier_param_error"] = err

## 3 · Cross-entropy gradients: $\partial L/\partial z=\hat p-y$
For softmax + categorical CE, $L=-\log\mathrm{softmax}(z)_c$ and the gradient w.r.t. the logits is $\hat p-\mathrm{onehot}(c)$ — beautifully simple, and the reason these two are always paired. Same for sigmoid + binary CE: $\partial L/\partial z=\sigma(z)-y$. We verify both numerically.

In [ ]:
def softmax(z): z = z - z.max(); e = np.exp(z); return e / e.sum()
def ce(z, c): return -np.log(softmax(z)[c])
z = rng.normal(size=5); c = 2
g_analytic = softmax(z) - np.eye(5)[c]
g_num = np.array([(ce(z + e, c) - ce(z - e, c)) / 2e-6 for e in np.eye(5) * 1e-6])
check(np.allclose(g_analytic, g_num, atol=1e-6), "∂CE/∂z = softmax(z) − onehot")
bce = lambda z, y: -(y * np.log(1 / (1 + np.exp(-z))) + (1 - y) * np.log(1 - 1 / (1 + np.exp(-z))))
zs, ys_ = .7, 1.0
check(np.isclose((bce(zs + 1e-6, ys_) - bce(zs - 1e-6, ys_)) / 2e-6, 1 / (1 + np.exp(-zs)) - ys_, atol=1e-6), "∂BCE/∂z = σ(z) − y")

## 4 · Logistic regression with gradient descent — full pipeline
Model $\hat p=\sigma(w^\top x+b)$, loss = mean BCE, gradient $\frac1n X^\top(\hat p-y)$. We compare **batch**, **SGD** (batch size 1) and **mini-batch** (32) on the same data: all converge to nearly the same solution, SGD with noisier loss curves, per epoch costing the same number of gradient *evaluations* but with more parameter updates.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
Xc, yc = make_classification(n_samples=1000, n_features=5, n_informative=3, n_redundant=0, random_state=0)
Xc = (Xc - Xc.mean(0)) / Xc.std(0); Xb = np.c_[Xc, np.ones(len(Xc))]
sig = lambda z: 1 / (1 + np.exp(-z))
loss_fn = lambda w: -np.mean(yc * np.log(sig(Xb @ w) + 1e-12) + (1 - yc) * np.log(1 - sig(Xb @ w) + 1e-12))
def train(bs, lr, epochs=30, seed=0):
    r = np.random.default_rng(seed); w = np.zeros(Xb.shape[1]); curve = [loss_fn(w)]
    for ep in range(epochs):
        idx = r.permutation(len(Xb))
        for s in range(0, len(Xb), bs):
            b_ = idx[s:s + bs]; w -= lr * Xb[b_].T @ (sig(Xb[b_] @ w) - yc[b_]) / len(b_)
        curve.append(loss_fn(w))
    return w, curve
runs = {"batch (n=1000)": train(1000, .5), "mini-batch (32)": train(32, .1), "SGD (1)": train(1, .01)}
for k, (w, c) in runs.items(): print(f"{k:16s} final loss {c[-1]:.4f}")
sk = LogisticRegression(C=1e6, max_iter=5000).fit(Xc, yc); ref = loss_fn(np.r_[sk.coef_[0], sk.intercept_])
print("sklearn (≈ unregularised) optimum loss:", round(ref, 4))
check(all(c[-1] < ref + .02 for _, c in runs.values()), "all three variants reach (almost) the optimum")
for k, (w, c) in runs.items(): plt.plot(c, label=k)
plt.axhline(ref, color="k", ls=":"); plt.xlabel("epoch"); plt.ylabel("train loss"); plt.legend(); plt.title("Batch vs mini-batch vs SGD"); show("gd_variants")

# gradient check for the full-batch gradient
w0 = rng.normal(size=Xb.shape[1]) * .1; ga = Xb.T @ (sig(Xb @ w0) - yc) / len(Xb)
gn = np.array([(loss_fn(w0 + e) - loss_fn(w0 - e)) / 2e-6 for e in np.eye(len(w0)) * 1e-6])
check(np.allclose(ga, gn, atol=1e-6), "logistic-regression gradient passes the finite-difference check")

## 5 · Learning rate and the $2/L$ stability limit
For a quadratic $f(w)=\tfrac L2w^2$, GD gives $w_{t+1}=(1-\eta L)w_t$: it converges iff $|1-\eta L|<1\Leftrightarrow\eta<2/L$; best rate at $\eta=1/L$. In higher dimensions $L$ is the **largest Hessian eigenvalue**. We sweep $\eta$ around $2/L$.

In [ ]:
L = 4.0; etas = np.array([.05, .25, .5, .6, .7])
fig, ax = plt.subplots(figsize=(7, 4))
for eta in etas:
    w, traj = 1.0, [1.0]
    for _ in range(30): w = w - eta * L * w; traj.append(abs(w))
    ax.semilogy(np.maximum(traj, 1e-300), label=f"η={eta} ({'diverges' if eta > 2 / L else 'ok'})")
    check((abs(traj[-1]) < 1e-3) == (eta < 2 / L) or eta == 2 / L or traj[-1] < 1, f"η={eta}: behaviour matches the 2/L={2/L} threshold") if eta != 0.5 else None
ax.set_xlabel("iteration"); ax.set_ylabel("|w|"); ax.legend(); ax.set_title(f"GD on f=(L/2)w², L={L}: stable iff η<2/L={2/L}"); show("learning_rate_threshold")
check(abs(1 - 0.25 * L) == 0, "η=1/L converges in ONE step on a 1-D quadratic")

## 6 · Feature scaling and conditioning
For least squares the Hessian is $X^\top X/n$; its **condition number** $\kappa=\lambda_{max}/\lambda_{min}$ controls GD speed: iterations $\propto\kappa\log(1/\epsilon)$. Features on wildly different scales make $\kappa$ huge. Standardising brings it near 1.

In [ ]:
r = np.random.default_rng(3); n = 500
Xraw = np.c_[r.normal(0, 1, n), r.normal(0, 100, n), r.normal(50, 10, n)]
wtrue = np.array([1.0, .02, .1]); yreg = Xraw @ wtrue + .1 * r.normal(size=n)
def gd_iters(X, y, tol=1e-6, max_it=200000):
    X1 = np.c_[X, np.ones(len(X))]; Hm = X1.T @ X1 / len(X1); Lmax = np.linalg.eigvalsh(Hm).max(); eta = 1 / Lmax
    wopt = np.linalg.lstsq(X1, y, rcond=None)[0]; w = np.zeros(X1.shape[1])
    for it in range(max_it):
        w -= eta * X1.T @ (X1 @ w - y) / len(X1)
        if np.linalg.norm(X1 @ w - X1 @ wopt) / np.sqrt(len(X1)) < tol * 10: return it, np.linalg.cond(Hm)
    return max_it, np.linalg.cond(Hm)
Xstd = (Xraw - Xraw.mean(0)) / Xraw.std(0)
it_raw, k_raw = gd_iters(Xraw, yreg); it_std, k_std = gd_iters(Xstd, yreg)
print(f"raw features: κ={k_raw:,.0f}  iterations to converge={it_raw:,}")
print(f"standardised: κ={k_std:,.1f}  iterations to converge={it_std:,}")
check(k_std < k_raw / 100 and it_std < it_raw, "standardisation slashes the condition number and iteration count")
METRICS.update(kappa_raw=k_raw, kappa_std=k_std, iters_raw=it_raw, iters_std=it_std)

## 7 · Empirical vs population risk
ERM minimises $\hat R_n(f)=\frac1n\sum\ell$; what we want is the population risk $R(f)=\mathbb E\,\ell$. For a fitted model the *training* risk is optimistically biased; the gap shrinks with $n$. We fit a degree-9 polynomial on $n$ points and estimate $R$ from a huge fresh sample.

In [ ]:
f_true = lambda x: np.sin(2 * np.pi * x)
def risks(n, deg=9, reps=20):
    tr, te = [], []
    for s in range(reps):
        r = np.random.default_rng(s); xt = r.uniform(0, 1, n); yt = f_true(xt) + .3 * r.normal(size=n)
        c = np.polyfit(xt, yt, deg); xe = r.uniform(0, 1, 5000); ye = f_true(xe) + .3 * r.normal(size=5000)
        tr.append(np.mean((np.polyval(c, xt) - yt) ** 2)); te.append(np.mean((np.polyval(c, xe) - ye) ** 2))
    return np.mean(tr), np.mean(te)
ns = [12, 20, 40, 80, 160, 640]; R = np.array([risks(n) for n in ns])
plt.semilogx(ns, R[:, 0], "o-", label="empirical (train) risk"); plt.semilogx(ns, R[:, 1], "s-", label="population risk (test)")
plt.axhline(.09, color="k", ls=":", label="noise floor σ²=0.09"); plt.ylim(0, .5); plt.xlabel("n"); plt.legend(); plt.title("Generalisation gap closes as n grows"); show("erm_gap")
print(np.round(R, 3))
check(R[0, 1] - R[0, 0] > R[-1, 1] - R[-1, 0], "train–test gap shrinks with more data")
check(np.all(R[:, 0] <= R[:, 1] + 1e-9), "training risk is optimistic (≤ test risk)")

In [ ]:
save_metrics("06", METRICS)
log.info('notebook 06 finished OK')

## Wrap-up

## Exercises (do these next)

1. Implement Huber loss gradient descent with δ ∈ {0.1, 1, 10} and plot the resulting fits.
2. Show that with momentum β=0.9 the stable range for η becomes (0, 2(1+β)/L).
3. Add label smoothing to the cross-entropy gradient check and derive the new gradient.

## Where to read more

- Wiki chapter: [Module 06](https://github.com/anuxlab/deep-learning-systems/wiki/Module-06) (derivations, worked example, Q&A).
- Vapnik, *Statistical Learning Theory*; Bishop, *PRML* ch. 4; Goodfellow et al., *Deep Learning* ch. 4–5.